# RFM + K-Means Customer Segmentation – Solution

**Short name (GitHub):** `CustSeg`

Worked answers for `CustSeg_Practice_Skeleton.ipynb`. Numbers below are from **this extract** (36,110 rows → 3,496 customers, £572,260), not from the full 541k-row UCI file.



## Inline cheat-sheet (keep this cell visible)

See also **`CustSeg_Cheatsheet.docx`**.

| Item | Formula / code |
|------|----------------|
| Line item vs customer | row in `online_retail.csv` = one SKU on an invoice; RFM row = one `CustomerID` |
| TotalSum | \(Q \times P\) after `Quantity > 0` (and usually `UnitPrice > 0`) |
| Snapshot | \(\text{snapshot} = \max(\text{InvoiceDate}) + 1\text{ day}\) |
| Recency | \((\text{snapshot} - \max_i t_i).\text{days}\) — smaller is warmer |
| Frequency | `nunique(InvoiceNo)`, **not** line-item `count` |
| Monetary | \(\sum Q\cdot P\) over the window |
| Skew fix | `np.log1p` on R, F, M **before** scaling |
| Scale | \(x'=(x-\mu)/\sigma\) on the log frame; keep \(\mu,\sigma\) for new customers |
| Inertia | \(J=\sum_i\|x_i-c_{\ell_i}\|^2\) on the **scaled** matrix |
| Elbow / sil. | plot \(J(k)\); confirm with silhouette; this extract likes \(k=4\) |
| Name bins | groupby Cluster → **median** R/F/M in original units |
| PCA | a slide of the 3-D scaled space; PC1 ≈ value, PC2 ≈ recency |

**Order:** clean → RFM → log1p → scale → choose \(k\) → fit → profile originals → PCA last.



## Desired outcome

![flowchart](custseg_flowchart.png)

1. Load `data/online_retail.csv`. Drop missing `CustomerID`. Keep `Quantity > 0` (and `UnitPrice > 0` for a strict sales book).
2. `TotalSum = Quantity * UnitPrice`. Parse `InvoiceDate` with `%d.%m.%Y %H:%M`. Cast `CustomerID` to int.
3. `snapshot_date = max(InvoiceDate) + 1 day`. Aggregate Recency / Frequency / Monetary per customer.
4. `log1p` the three columns, then `StandardScaler`. Do not overwrite the original RFM table.
5. Elbow + silhouette for \(k=1\ldots10\). Fit K-Means at the chosen \(k\) (we use 4).
6. Write `rfm["Cluster"] = labels`. Profile **medians** in original units and name the bins.
7. PCA to 2-D is a picture, not the model. Alternates, more practice, then turn the simulation knobs.



## 0. Packages


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from sklearn.cluster import KMeans, MiniBatchKMeans, AgglomerativeClustering
    from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler
    from sklearn.decomposition import PCA
    from sklearn.metrics import silhouette_score
    HAS_SK = True
except ImportError:
    HAS_SK = False
    print("sklearn not found — install scikit-learn to run the clustering cells.")

import CustSeg as cs

plt.rcParams["figure.figsize"] = (8, 4.5)
np.set_printoptions(precision=3, suppress=True)
pd.set_option("display.max_columns", 20)
print("sklearn", HAS_SK, "| CustSeg helpers ready")


## 1. Why RFM then K-Means

Retail invoices are the wrong grain for a segment. RFM collapses the book to one row per `CustomerID`. K-Means then needs a roughly spherical cloud — hence log1p on the whale tail and a z-score before `.fit`.



## 2. Load and clean


In [ ]:
raw = pd.read_csv("data/online_retail.csv")
print("raw shape", raw.shape)
print(raw.dtypes)
print("missing\n", raw.isna().sum())
print(raw[["Quantity", "UnitPrice"]].describe())
print("C-invoices", raw["InvoiceNo"].astype(str).str.startswith("C").sum())
print(raw["Country"].value_counts().head())


In [ ]:
sales = raw.dropna(subset=["CustomerID"]).copy()
sales = sales[sales["Quantity"] > 0]
sales = sales[sales["UnitPrice"] > 0]
sales["TotalSum"] = sales["Quantity"] * sales["UnitPrice"]
sales["InvoiceDate"] = pd.to_datetime(sales["InvoiceDate"], format="%d.%m.%Y %H:%M")
sales["CustomerID"] = sales["CustomerID"].astype(int)

print("clean shape", sales.shape)
print("negative qty left", int((sales["Quantity"] < 0).sum()))
print("missing CID", int(sales["CustomerID"].isna().sum()))
print("revenue", round(sales["TotalSum"].sum(), 2))
print("customers", sales["CustomerID"].nunique(), "invoices", sales["InvoiceNo"].nunique())
print("window", sales["InvoiceDate"].min(), "→", sales["InvoiceDate"].max())


## 3. Look at the book before you cluster


In [ ]:
by_cty = sales.groupby("Country")["TotalSum"].sum().sort_values(ascending=False)
print(by_cty.head(6).round(2))
print("UK share", round(by_cty.get("United Kingdom", 0) / by_cty.sum(), 3))

fig, ax = plt.subplots()
clipped = sales["TotalSum"].clip(upper=sales["TotalSum"].quantile(0.99))
ax.hist(clipped, bins=40, color="#4c78a8", edgecolor="white")
ax.set_title("Line-item TotalSum (99th pct clip) — not yet RFM Monetary")
ax.set_xlabel("£")
plt.show()


## 4. Build the RFM table


In [ ]:
snapshot = sales["InvoiceDate"].max() + pd.Timedelta(days=1)
print("snapshot", snapshot)

rfm = sales.groupby("CustomerID").agg(
    Recency=("InvoiceDate", lambda x: (snapshot - x.max()).days),
    Frequency=("InvoiceNo", "nunique"),
    Monetary=("TotalSum", "sum"),
).reset_index()

print(rfm.head())
print(rfm[["Recency", "Frequency", "Monetary"]].describe().round(2))
print("skew", rfm[["Recency", "Frequency", "Monetary"]].skew().round(2).to_dict())
print("n", len(rfm))


## 5. Log1p then scale


In [ ]:
rfm_log = rfm[["Recency", "Frequency", "Monetary"]].apply(np.log1p)
print("log skew", rfm_log.skew().round(2).to_dict())

scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_log)
print("scaled mean", rfm_scaled.mean(axis=0))
print("scaled std ", rfm_scaled.std(axis=0))


## 6. Elbow and silhouette


In [ ]:
ks = list(range(1, 11))
inertias, sils = [], [np.nan]
for k in ks:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    lab = km.fit_predict(rfm_scaled)
    inertias.append(km.inertia_)
    if k >= 2:
        sils.append(silhouette_score(rfm_scaled, lab))

print("inertia", [round(v, 1) for v in inertias])
print("sil    ", [None if np.isnan(v) else round(v, 3) for v in sils])

fig, ax1 = plt.subplots()
ax1.plot(ks, inertias, "o-", color="#4c78a8", label="inertia")
ax1.axvline(4, color="#e15759", ls="--", label="k=4")
ax1.set_xlabel("k")
ax1.set_ylabel("inertia")
ax2 = ax1.twinx()
ax2.plot(ks, sils, "s--", color="#59a14f", label="silhouette")
ax2.set_ylabel("silhouette")
h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, loc="center right")
ax1.set_title("Elbow + silhouette on log-scaled RFM")
ax1.grid(True, alpha=0.3)
plt.show()


## 7. Fit K-Means and attach labels


In [ ]:
k_optimal = 4
kmeans_final = KMeans(n_clusters=k_optimal, random_state=42, n_init=10)
kmeans_final.fit(rfm_scaled)
rfm["Cluster"] = kmeans_final.labels_
print(rfm["Cluster"].value_counts().sort_index().to_dict())


## 8. Profile in original units and name the bins


In [ ]:
cluster_summary = (
    rfm.groupby("Cluster")[["Recency", "Frequency", "Monetary"]]
    .agg(["mean", "median", "count"])
    .round(1)
)
print(cluster_summary)

share_n = rfm.groupby("Cluster").size() / len(rfm)
share_£ = rfm.groupby("Cluster")["Monetary"].sum() / rfm["Monetary"].sum()
print("% customers\n", (100 * share_n).round(1))
print("% revenue\n", (100 * share_£).round(1))

# Seed 42 on this extract:
# 2 = Champions (low R, high F/M, 16.6% cust / 63.7% £)
# 3 = Recent occasional (low R, low F/M)
# 0 = Slipping mid-value (mid R, mid F/M)
# 1 = Lost / hibernating (high R, F≈1, tiny £)
NAME = {
    2: "Champions",
    3: "Recent occasional",
    0: "Slipping mid-value",
    1: "Lost / hibernating",
}
rfm["Segment"] = rfm["Cluster"].map(NAME)
print(rfm.groupby("Segment")[["Recency", "Frequency", "Monetary"]].median().round(1))


## 9. PCA is a slide, not the model


In [ ]:
pca = PCA(n_components=2, random_state=42)
components = pca.fit_transform(rfm_scaled)
pca_df = pd.DataFrame(components, columns=["PC1", "PC2"])
pca_df["Cluster"] = rfm["Cluster"].to_numpy()
pca_df["Segment"] = rfm["Segment"].to_numpy()

print("explained variance", pca.explained_variance_ratio_.round(3),
      "sum", round(pca.explained_variance_ratio_.sum(), 3))
loadings = pd.DataFrame(
    pca.components_.T,
    index=["Recency", "Frequency", "Monetary"],
    columns=["PC1", "PC2"],
).round(3)
print(loadings)

fig, ax = plt.subplots(figsize=(8, 6))
for seg, g in pca_df.groupby("Segment"):
    ax.scatter(g["PC1"], g["PC2"], s=16, alpha=0.65, label=seg)
ax.set_xlabel("PC1  value / engagement")
ax.set_ylabel("PC2  recency contrast")
ax.set_title("Customer segments via PCA (scaled RFM)")
ax.legend()
ax.grid(True, alpha=0.25)
plt.show()


## 10. Alternate code that reaches the same idea


In [ ]:
# A. fit_predict
alt_labels = KMeans(n_clusters=4, random_state=42, n_init=10).fit_predict(rfm_scaled)
print("A fit_predict agrees", np.array_equal(alt_labels, rfm["Cluster"].to_numpy()))

# B. RobustScaler after the same log1p
X_rob = RobustScaler().fit_transform(rfm_log)
lab_rob = KMeans(n_clusters=4, random_state=42, n_init=10).fit_predict(X_rob)
print("B RobustScaler sizes", pd.Series(lab_rob).value_counts().sort_index().to_dict())

# C. quintile RFM card (R inverted)
rfm_q = cs.rfm_quintile_scores(rfm)
print("C RFM score head")
print(rfm_q[["CustomerID", "R", "F", "M", "RFM"]].head())
print("C champions-like (R>=4 & F>=4 & M>=4)", int(((rfm_q.R >= 4) & (rfm_q.F >= 4) & (rfm_q.M >= 4)).sum()))

# D. agglomerative
lab_agg = AgglomerativeClustering(n_clusters=4).fit_predict(rfm_scaled)
print("D Agglomerative sizes", pd.Series(lab_agg).value_counts().sort_index().to_dict())

# E. SVD PCA
Xc = rfm_scaled - rfm_scaled.mean(0)
_, S, Vt = np.linalg.svd(Xc, full_matrices=False)
Z_svd = Xc @ Vt[:2].T
print("E SVD var share", ((S[:2] ** 2) / (S ** 2).sum()).round(3))


## 11. More practice


In [ ]:
# P1. UK only
uk_sales = sales[sales["Country"] == "United Kingdom"]
rfm_uk = cs.build_rfm(uk_sales, snapshot)
_, X_uk, _, _ = cs.log_scale(rfm_uk)
_, lab_uk = cs.fit_kmeans(X_uk, k=4)
print("P1 UK customers", len(rfm_uk))
print(cs.profile_clusters(rfm_uk, lab_uk))

# P2. trailing 180 days
cut = snapshot - pd.Timedelta(days=180)
recent_sales = sales[sales["InvoiceDate"] >= cut]
rfm_180 = cs.build_rfm(recent_sales, snapshot)
print("P2 180d customers", len(rfm_180), "revenue", round(rfm_180["Monetary"].sum(), 2))

# P3. one new customer scored with the training scaler
new = pd.DataFrame([[7, 6, 400.0]], columns=["Recency", "Frequency", "Monetary"])
new_scaled = scaler.transform(np.log1p(new))
print("P3 predicted cluster", int(kmeans_final.predict(new_scaled)[0]))

# P4. drop Frequency
X_rm = rfm_scaled[:, [0, 2]]
lab_rm = KMeans(n_clusters=4, random_state=42, n_init=10).fit_predict(X_rm)
print("P4 Recency+Monetary sizes", pd.Series(lab_rm).value_counts().sort_index().to_dict())


## 12. Simulation — turn the knobs


In [ ]:
K = 4
N = None
NOISE = 0.0
N_INIT = 10
SEED = 42

print(cs.simulate(rfm_scaled, k=K, n=N, noise=NOISE, n_init=N_INIT, seed=SEED))

rows = []
for k in range(2, 9):
    for noise in (0.0, 0.15, 0.30):
        rows.append(cs.simulate(rfm_scaled, k=k, n=N, noise=noise, n_init=N_INIT, seed=SEED))
sim = pd.DataFrame([r.__dict__ for r in rows])
print(sim.pivot(index="k", columns="noise", values="silhouette").round(3))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for noise, g in sim.groupby("noise"):
    axes[0].plot(g["k"], g["inertia"], "o-", label=f"noise={noise}")
    axes[1].plot(g["k"], g["silhouette"], "s-", label=f"noise={noise}")
axes[0].set_title("Inertia vs k")
axes[1].set_title("Silhouette vs k")
for ax in axes:
    ax.set_xlabel("k")
    ax.legend()
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Audience rewrite (Jočys checklist + McMurrey types)

| Audience | What they need | One sentence they should hear |
|----------|----------------|-------------------------------|
| Expert (CRM scientist) | inertia, silhouette, loadings, log+scale order | k=4 is compatible with the elbow; PC1 (71%) is value, PC2 (21%) is recency; name bins from medians. |
| Technician (campaign ops) | a four-row treatment table and the query keys | Champions = cluster 2 on this seed; suppress win-back mail to cluster 1 until a human reviews the offer. |
| Executive (CMO / retail) | concentration, not eigenvalues | 17% of customers produce 64% of observed revenue; the slipping mid-value bin is the cheapest win-back. |
| Nonspecialist | a store-floor analogy | We sorted shoppers by *how lately, how often, how much* — not by postcode or first name. |



## What this model can and cannot do

**Can** — group RFM rows, show value concentration, give ops four named bins.

**Cannot** — forecast the next basket, mint a loyalty tier, travel to another banner without refitting, treat a cluster id as a permanent badge.

**Top uses:** CRM design, win-back vs nurture, wholesale-vs-retail separation, board concentration slides.
**Anti-uses:** automated markdowns, scoring with a different scaler, clustering raw unlogged Monetary.



## Next steps

- Add country or channel as a scaled dummy.
- Try k=3 if ops will only fund three treatments.
- Cap Monetary at the 99th percentile before log1p.
- Read `CustSeg_Strategy_Guide.docx` before cloning onto another ledger.

